# 02 Design Preparation

## Goal
Prepare the 1LYZ structure for real protein design workflows.

## Purpose
- load the selected target structure
- confirm the structure used for design
- export a clean structure file for downstream tools
- separate design preparation from exploratory structural analysis

## Version context
This notebook marks the transition from v0 structural analysis to v4 real design workflow.

***Load the structure again in the new notebook***

In [2]:
from pathlib import Path
from Bio.PDB import PDBParser

pdb_path = Path("../data/raw/1LYZ.pdb")

parser = PDBParser(QUIET=True)
structure = parser.get_structure("1LYZ", pdb_path)

print("Loaded structure:", structure)
print("Using file:", pdb_path)

Loaded structure: <Structure id=1LYZ>
Using file: ../data/raw/1LYZ.pdb


***Export a clean structure file***

In [3]:
from Bio.PDB import PDBIO
from pathlib import Path

output_path = Path("../data/design/1LYZ_clean.pdb")

io = PDBIO()
io.set_structure(structure)
io.save(str(output_path))

print("Saved cleaned structure to:", output_path)

Saved cleaned structure to: ../data/design/1LYZ_clean.pdb


***Add a quick existence check***

In [6]:
print("Cleaned file exists:", output_path.exists())

Cleaned file exists: True


## Clean Design Input

A cleaned copy of the 1LYZ structure was exported for downstream design workflows.

### Why this matters
- preserves the original raw input separately
- defines the exact structure used as the design backbone
- improves reproducibility for real design tools

### Important limitation
This export step does not yet remove non-protein residues or apply tool-specific filtering. It only establishes a clean structural design starting point.

***verify the structure again***

In [7]:
model = structure[0]
chains = list(model.get_chains())

print("Chains found:", [chain.id for chain in chains])

Chains found: ['A']


In [8]:
from Bio.PDB.Polypeptide import is_aa

residue_count = sum(
    1 for residue in model.get_residues()
    if is_aa(residue, standard=True)
)

print("Number of amino-acid residues:", residue_count)

Number of amino-acid residues: 129


***Export a protein-only design file***

In [9]:
from Bio.PDB import Select, PDBIO
from Bio.PDB.Polypeptide import is_aa
from pathlib import Path

class ProteinOnly(Select):
    def accept_residue(self, residue):
        return is_aa(residue, standard=True)

protein_only_path = Path("../data/design/1LYZ_clean_protein_only.pdb")

io = PDBIO()
io.set_structure(structure)
io.save(str(protein_only_path), ProteinOnly())

print("Saved protein-only structure to:", protein_only_path)

Saved protein-only structure to: ../data/design/1LYZ_clean_protein_only.pdb


***verify exported file***

In [11]:
print("Protein-only file exists:", protein_only_path.exists())

Protein-only file exists: True


***quick file-size sanity check - the protein-only file is slightly smaller, that is expected.***

In [14]:
clean_path = Path("../data/design/1LYZ_clean.pdb")

print("Clean file size (bytes):", clean_path.stat().st_size)
print("Protein-only file size (bytes):", protein_only_path.stat().st_size)

Clean file size (bytes): 89351
Protein-only file size (bytes): 81170
